In [1]:
# Imports
import gc

from numpy.f2py.auxfuncs import throw_error

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert


ss = hf.settings_dict()

In [2]:
tmin = 0.7
tmax = 3.7



In [3]:
for subject_index in ss['subject_idx_list']:
    max_mean_voxels = []
    for event_id in ss['event_id_list']:
        event_name = str(event_id)
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]

        save_dir = Path(ss['hilbert_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # pick highest mean amplitude voxel across all trials
        amplitude_data = []
        for i in range(ss['n_trials']):

            stc_file = Path(ss['stc_dir']) / subject / event_name / f"{subject}-{i}-vol.stc"

            stc = mne.read_source_estimate(stc_file)
            stc.crop(tmin=tmin, tmax=tmax)

            analytic_signal = hilbert(stc.data, axis=1)
            amplitude = np.abs(analytic_signal)

            amplitude_data.append(amplitude)

        stacked_amplitude_data = np.stack(amplitude_data)

        mean_amplitude_data = np.mean(stacked_amplitude_data, axis=0)

        mean_amp = np.mean(mean_amplitude_data, axis=1)

        max_mean_voxel = np.argmax(mean_amp)

        max_mean_voxels.append([max_mean_voxel, mean_amp[max_mean_voxel]])

    ref_voxel = max(max_mean_voxels, key=lambda x: x[1])[0]

    print(f"Picking voxel {ref_voxel} as a reference.")
    # loop over each event type
    for event_id in ss['event_id_list']:
        event_name = str(event_id)
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)


        for i in range(ss['n_trials']):

            stc_file = Path(ss['stc_dir']) / subject / event_name / f"{subject}-{i}-vol.stc"

            stc = mne.read_source_estimate(stc_file)
            cropped_stc = stc.copy()

            # hilbert transform the stc

            analytic_signal = hilbert(cropped_stc.data, axis=1)
            phase = np.angle(analytic_signal).astype(float)

            # save the hilbert transformations in a new stc

            hil_stc = cropped_stc.copy()
            hil_stc.data = phase

            hil_stc.save(save_dir / f"{subject}-trial-{i}-hilbert-vol.stc" , overwrite=True)

            # now save the hilbert transformed references

            times = cropped_stc.times
            voxel_phase = phase[ref_voxel, :]

            # read the retina csv
            retina_file = Path(ss['stc_dir']) / subject / event_name / f"{subject}-trial-{i}-retina.csv"

            retina_df = pd.read_csv(retina_file)

            retina_df_cropped = retina_df

            analytic_signal_retina = hilbert(retina_df_cropped["amplitude"])
            retina_phase = np.angle(analytic_signal_retina).astype(float)

            # Create dataframe
            df = pd.DataFrame({
                "time_s": times,
                "retina_phase": retina_phase,
                f"{str(ref_voxel)}_voxel_phase": voxel_phase,
            })

            # Save to CSV
            df.to_csv(save_dir / f"{subject}-trial-{i}-reference.csv" , index=False)


gc.collect()


Picking voxel 336 as a reference.
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk..

0